Stock Market Prediction using Random Forest

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1THfR6xQbJf3C1whnGg3tZ2ZLw3rji49l

# Dataset Introduction

S&P 500 (Standard & Poor’s 500)
The S&P 500 is a stock market index that tracks the performance of 500 of the largest publicly traded companies in the United States. It is widely regarded as one of the best representations of the overall U.S. stock market and economy. The index is market capitalization-weighted, meaning larger companies (like Apple, Microsoft, and Amazon) have a greater influence on its movement.

GSPC (^GSPC)
^GSPC is the ticker symbol for the S&P 500 Index when tracking it in financial data sources like Yahoo Finance. It provides historical and real-time data for the S&P 500, including:

- Open, high, low, and closing prices

- Trading volume

- Market trends over time

# Downloading S&P500 PRICE DATA

In [ ]:
# calls yahoo-finance api to download daily stock and index prices
import yfinance as yf

# initialising ticker class
# will help us download price history for a single symbol
sp500 = yf.Ticker("^GSPC")

#GSPC symbol is for S&P500 index

# query all data since the index was created
sp500 = sp500.history(period="max")

# now sp500 is a pandas dataframe
# it has datewise prices (only of working days) for opening, max and closing prices as well as volumes traded
# we will use this to predict if the next day the price will rise or no
# we won't be needing Dividends and Stock Spilts columns

sp500

# check index of dataset
sp500.index

# we will be using this index to SLICE and INDEX the dataframe easily

(Extra) Index column in Pandas

Index Column in a DataFrame
In pandas, the index column of a DataFrame is a unique identifier for each row. It helps in fast lookups, data alignment, and efficient operations on large datasets.

- It can be a single column or a multi-level (hierarchical) index.

- By default, pandas assigns an index as 0, 1, 2, … (integer-based index).

Why Use an Index?

- Faster Lookups → Access rows quickly using .loc[ ]

- Time Series Analysis → Useful for stock market data with date-based indexing

- Data Alignment → Helps merge and join datasets efficiently

In [ ]:
import pandas as pd

data = {'Date': ['2024-04-01', '2024-04-02', '2024-04-03'],
        'Close Price': [5000, 5050, 5100]}

df = pd.DataFrame(data)

# Default indexing:
print(df)

# Choose a column as an index (column)
df.set_index('Date', inplace=True)
print(df)

inplace=True: Modifies the DataFrame directly instead of returning a new DataFrame.

In [ ]:
# De-seletc a column as an index (column)
df.reset_index(inplace=True)
print(df)

# Visualisations

In [ ]:
# Youtube vid
# y-axis: closing prices, x-axis: index column (dates)
sp500.plot(y = "Close", use_index=True)

# colab AI generated

from matplotlib import pyplot as plt
sp500['Close'].plot(kind='line', figsize=(8, 4), title='Close')
plt.gca().spines[['top', 'right']].set_visible(False)

# Close vs Volume (colab AI)
from matplotlib import pyplot as plt
sp500.plot(kind='scatter', x='Close', y='Volume', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

# Pre-processing and Data-cleaning

// THOUGHT PROCESS //

Columns like
- Dividends
- Stock Splits

are less relevant for tracking an index (and more relevant to predict stock prices)

Stock = High-risk, high-reward, company-specific

Index = Lower risk, diversified, tracks the overall market

Example Scenario

- If you invest in Apple (AAPL), your profit/loss depends only on Apple’s performance.

- If you invest in the S&P 500, your returns depend on the 500 biggest U.S. companies (including Apple), reducing risk.

In [ ]:
# delete the two columns
del sp500['Dividends']
del sp500['Stock Splits']

# OR drop the columns
# sp500 = sp500.drop(columns=['Dividends', 'Stock Splits'])

sp500

// THOUGHT PROCESS

We don't want to deal with all rows of data to account for recency bias in trends

In [ ]:
sp500 = sp500.loc["1998-01-01": ].copy()
sp500

WHY .copy() ??

- It creates a completely new and independent DataFrame.

- Without .copy(), modifying sp500 might unintentionally modify the original dataset.

- Avoids SettingWithCopyWarning:
If you modify the new sp500 without .copy(), you might get a SettingWithCopyWarning (it happens when pandas is unsure whether your changes will affect the original DataFrame or not)

# Selecting target column for ML

// THOUGHT PROCESS

Instead of trying to predict exact price on a day, we will try to predict if price will go up or down

In [ ]:
# create a new column Tomorrow which will have the next day's closing price
sp500['Tomorrow'] = sp500['Close'].shift(-1)
sp500

The .shift(-1) method moves all values in the 'Close' column up by one row.

In [ ]:
# now set up target (this is what we try to predict in Machine Learning)

sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)          # convert logic to number
sp500

# Baseline Model (Random Forest)

WHY?

- Good default for most ML
- Harder to overfit than other models: train some independant decision trees, with randomized parameters, and average results from all trees
- Runs relatively fast
- Can pick up non-linear patterns

The Random Forest algorithm is inherently random because it involves two main sources of randomness:

- Bootstrap Sampling (if bootstrap=True)    
    - Each tree in the forest is trained on a random subset of the data (sampled with replacement).
        - If bootstrap=True, each tree in the Random Forest is trained on a random subset of the original dataset.
        - The subset is created by sampling with replacement, meaning some data points may appear multiple times, while others may be left out.

- Random Feature Selection at Each Split (max_features)
    - When a tree grows, it randomly selects a subset of features (instead of all features) at each node to determine the best split.
    - This makes the model more robust and diverse because each tree may make decisions based on a different set of features.

Random Forest parameters:

- n_estimators
    - Defines the number of decision trees in the forest.
    - A typical range is 10-500 depending on dataset size and complexity.
    - More trees: generally improve performance but increase computation time.

- min_samples_split
    - Minimum samples needed to split a node
    - Controls when a node can be split
    - Higher value = less splits (reduces overfitting)
    - prevents splitting too often based on outliers or noise (limits node creation), keeping the tree from growing too deep.

- min_samples_leaf
    - Minimum samples required in a leaf node
    - Ensures that each leaf has at least this many samples
    - Higher value = shallower trees (reduces variance)
    - ensures that leaves are not too small (prevents overly specific branches) that they capture noise.

- random_state
    - Seed for reproducibility; helpful when we are trying to improve a model overtime
    - If you set random_state=1 (or any fixed value):
        - The same training subsets and feature selections will be used each time.
        - The model's structure and predictions will be the same across different runs.
- criterion
    - Function to measure quality of a split ("gini" or "entropy")
    - "gini" is default, "entropy" is better for imbalanced classes
        - Entropy takes into account the logarithmic scale, which makes it more sensitive to imbalances. Entropy is better for imbalanced data because it gives more weight to the minority class, forcing the tree to focus on splitting to separate it.
        - Gini, on the other hand, is less sensitive to small differences in probability. It might not "care" as much about the small class and will treat them similarly to the large class.
- max_features
    - Number of features considered at each split
    - If we set it to a small number (e.g., the square root of total features), each tree will see different subsets of features and thus behave more differently.
    - This ensures the trees are not too similar (fewer overlaps), even if they are trained on the same data and controls randomness and diversity in trees
- max_depth
    - Maximum depth of each tree
    - Limits overfitting by restricting tree size
- bootstrap
    - Whether bootstrapped samples are used; True (default) helps reduce variance
    - When set to False:
        - for large datasets, where there’s a lot of data to work with, using bootstrapping may not introduce as much diversity as expected.
        - Random Forest is a variant of bagging (Bootstrap Aggregating), where bootstrapping is a core principle. However, some variants of bagging ( like PASTING ) do not involve bootstrapping, meaning no replacement is used, and each tree is trained on a unique subset of the data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

# Train and Test Split

# Basic baseline model
train = sp500[:-100]        # rest of (cropped) dataset
test = sp500[-100:]         # last 100 rows

predictor_columns = ["Close", "Volume", "Open", "High", "Low"]

# mention: what to make make predictions based on AND what to predict
model.fit(train[predictor_columns], train["Target"])

Cross validation won't work:
- doesn't maintain time-series nature of data
- leakage: predictions based on "seeing" future values
    - will lead to amazing training results but horrible testing accuracy

In [ ]:
# Testing model accuracy
from sklearn.metrics import precision_score
import pandas as pd

predictions = model.predict(test[predictor_columns])

# Convert NumPy to Pandas Series bc easier to work with
# maintain same index as test dataset
predictions = pd.Series(predictions, index=test.index)
predictions

precision_score(test["Target"], predictions)  # which 2 columns to compare

# Plot predictions

# concatenate our actual vs predicted values
# axis = 1: treat each of these parameters inside concat function as columns in our dataset
combined = pd.concat([test["Target"], predictions], axis=1)
combined.plot()

# orange our, blue original

# Backtesting System

Improved system for making Train and Test datsets

In [ ]:
# culmination of everything in baseline model 1

def predict(train, test, predictors, model):
    # model has been initialised as Random Forest
    # specify which columns to use for training and which columns for testing
    model.fit(train[predictors], train["Target"])

    # generate predictions using .predict() method of model
    # mention you want to use the came pedictors but on the test dataset
    preds = model.predict(test[predictors])

    # predictions is a NumPy array
    # convert it to dataframe and add to dataset as column-name Predictions
    preds = pd.Series(preds, index=test.index, name = "Predictions")

    # combine the actual Target values and predicted Target values for side-by-side comparison
    # axis = 1 bc side-by-side concatenation
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

// THOUGHT PROCESS (IMPORTANT STRATEGIC TAKEAWAY)

- Each year has roughly 250 trading days
- So we will train on data for the first 10 years (2500 data) to predict the 11th year (250 days after 2500 days)
- Then we will train on data for the first 11 years (2500+250) to predict the 12th year and so on..

- So slice 1
    - Train: data 0 to data 2500
    - Test: 250 after data 2500
- So slice 2
    - Train: data 0 to data (2500 + 250)
    - Test: 250 after data (2500 + 250)
- So slice 3
    - Train: data 0 to data (2500 + 250 + 250)
    - Test: 250 after data (2500 + 250 + 250)
- continue till last rows have been included as test limit

In [ ]:
# our function will need to work on a DATASET and according to some MODEL
# predictions will be made based on PREDICTORS (columns)
# we will use START and STEP values for slicing dataset into Test and Train

def backtesting(data, model, predictors, start = 2500, step = 250):
    all_predictions = []

    for i in range(start, data.shape[0], step):     # check theory above
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        # TO MAKE PREDICTIONS, YOU NEED: dataset (split into TRAIN and TEST), model, predictors columns

        all_predictions.append(predictions)
    return pd.concat(all_predictions)                               # taking all prediction dataframes and combining/concating into 1 dataframe

predictions = backtesting(sp500, model, predictor_columns)
predictions

# index column came bc it is helping mainatin order in original-data vs prediction-data

IMPORTANT: To check unique count of each type of classes predicted

- predictions is a data frame
- we will only focus on it's Predictions column bc we generated it
- .values_count() method

In [ ]:
predictions["Predictions"].value_counts()

// RESULT EVALUATION THOUGHT PROCESS

Now, to compare how good our results is,we need a benchmark. The benchmark used below is:
- if we woke up everyday and blindly decided to buy stocks in morning and sell them at closing, we calculate on what percentage of days would we be making a profit (around 54%)
- Now we see if our model can help us do a better job
    - i.e. on what percentage of days that the models predicts us to make a profit do we ACTUALLY make a profit (around 52%)

In [ ]:
# we are using this as the benchmark
predictions["Target"].value_counts()  / predictions.shape[0]

# when calculating precision, we need to mention which 2 columns we comparing
precision_score(predictions["Target"], predictions["Predictions"])

# Adding extra predictors

// THOUGHT PROCESS

This is an attempt to improve accuracy

Intuition:

If you're a natural analyst, you'd compare whether today's stock price is more or less than:
- last 2 day's average prices
- last week's (5 days) average prices
- last 3 month's (60 days) average prices
- last 1 year's (250 days) average prices
- last 4 year's (1000 days) average prices

So, we will check out some rolling averages (eg. mean close prices) at certain time horizons

A rolling average, also called a moving average,
- is a statistical technique used to
    - smooth out short-term fluctuations
    - highlight longer-term trends in data.
- It works by computing the average of a fixed-sized window of consecutive data points as it "rolls" through the dataset.
Then we will compare these mean prices back then with the current prices (using ratios)

We will also lookup trends (eg. frequency of price rise in past 'i' days)
- shifting the entire sp500 DataFrame first (not just "Target")
    - Shifts "Target" down by 1 first, so each row's rolling sum excludes its own value (uses past 'i' values).
    - This prevents leakage and seeing future values
- then applying the rolling sum for past 'i' days
- then check the new "Target column values"

In [ ]:
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for i in horizons:

    # calculate rolling_avg of last i (= 2, 5, ..) days
    rolling_averages = sp500.rolling(i).mean()                                  # rolling is process, mean is the calculation
    new_col1_name = f"Close ratio (prev {i} days)"
    sp500[new_col1_name] = sp500["Close"] / rolling_averages["Close"]
    new_predictors.append(new_col1_name)

    # check trends like in last i (= 2, 5, ..) days, how many times the price went up
    new_col2_name = f"Trend (prev {i} days)"
    sp500[new_col2_name] = sp500.shift(1).rolling(i).sum()["Target"]            # rolling is process, sum is the calculation
    new_predictors.append(new_col2_name)


sp500

# cleaning data to get rid of NaN values
# these are generated when calculating rolling averages when not enough rows above are found

sp500.dropna()

# Finetuning the baseline model parameters

- more trees for better averaging
- lower min_samples_split for deeper trees

In [ ]:
# Change 1

model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

When using classification models like Logistic Regression, Random Forest, XGBoost, etc., the predict_proba() method returns the probability distribution over all possible classes
- For binary classification (0 or 1), predict_proba() returns an array where:
    - [:, 0] gives the probability of class 0 (negative class) [generally not used]
    - [:, 1] gives the probability of class 1 (positive class).
- For multiclass (eg. n) classification, predict_proba() returns a n-column array:
    - [:, 0] → Probabilities of class 0
    - [:, 1] → Probabilities of class 1
    - [:, 2] → Probabilities of class 2

In [ ]:
# Change 2

# used inside backtesting() function
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])

    # to have more confidence in our classification ============================
    # .predict() uses a default threshold of probability > 0.5 to choose class

    # preds = model.predict_proba(test[predictors])[:, 1]
    # preds[preds >= 0.6] = 1
    # preds[preds < 0.6] = 0

    preds = (model.predict_proba(test[predictors])[:, 1] > 0.6).astype(int)

    #===========================================================================
    preds = pd.Series(preds, index=test.index, name = "Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

# Implement changes
predictions = backtesting(sp500, model, new_predictors)

Notice:

We are getting rid of initial predictor columns: Opening price, Closing Price, ...

This is because the newly generated features (ratios, trends) are more relevant and realistic for our predictions
- eg. what is open price yesterday says little about whether price will go up or down today

In [ ]:
# Evaluate results on new predictions

precision_score(predictions["Target"], predictions["Predictions"])

We see that
- compared to previous model's 52% precision, we achieve a 55% precision
- compared to our benchmark's 54% blind-test precision, we achieve a 55% precision

# How to further improve

Recommendations
- There are some index/stocks which are traded overnight (unlike just during US trading hours)
    - Find correlation if an index price changing in other part of world help predict S&P500 prices
- Include news, articles, general macroeconomic conditions (eg. interest rates, inflation)
- Include key components of S&P500 (e. key stocks, key sectors)
    - Find correlation that if tech sector is in a downturn, S&P500 price will go down in 6 months (not immediately)
- Increase resolution (eg. hourly data instead of daily data)